# IPL T20 Graph Demo

**Pipeline:** Synthetic data → Parquet → GCS → FalkorDB → Cypher

This notebook loads an **IPL T20 Cricket Graph** using direct Parquet upload — no Starburst needed.

Graph shape:
- **Team** nodes: `team_id`, `name`, `city`, `titles`
- **Player** nodes: `player_id`, `name`, `role`, `nationality`, `batting_style`
- **Game** nodes: `match_id`, `season`, `venue`, `winner_team_id`, `mom_player_id`
- **PLAYS_FOR** edges: Player → Team
- **PLAYED_IN** edges: Player → Match (with `runs`, `wickets` properties)
- **WON** edges: Team → Match

In [1]:
# ── Step 1 │ Setup & Imports ───────────────────────────────────────────────
# Services : Control Plane (localhost:8081)
# ──────────────────────────────────────────────────────────────────────────
import requests, json, time, os
import pandas as pd
from pathlib import Path

API = "http://localhost:8081"
H   = {"X-Username": "demo@example.com", "X-User-Role": "admin", "Content-Type": "application/json"}

def get(path):        r = requests.get(f"{API}{path}", headers=H);   r.raise_for_status(); return r.json()
def post(path, body): r = requests.post(f"{API}{path}", headers=H, json=body); r.raise_for_status(); return r.json()
def delete(path):     r = requests.delete(f"{API}{path}", headers=H); r.raise_for_status(); return r.json()

print(requests.get(f"{API}/health").json())

{'status': 'healthy', 'database': 'unknown', 'version': '0.1.0'}


## Step 1 — Generate synthetic IPL data

In [2]:
# ── Step 2 │ Generate Synthetic Data ──────────────────────────────────────
# Services : (local Python — no network)
# ──────────────────────────────────────────────────────────────────────────
# --- Teams ---
teams = pd.DataFrame([
    {"team_id": 1, "name": "Mumbai Indians",            "city": "Mumbai",    "titles": 5},
    {"team_id": 2, "name": "Chennai Super Kings",        "city": "Chennai",   "titles": 5},
    {"team_id": 3, "name": "Royal Challengers Bengaluru","city": "Bengaluru", "titles": 0},
    {"team_id": 4, "name": "Kolkata Knight Riders",      "city": "Kolkata",   "titles": 3},
    {"team_id": 5, "name": "Delhi Capitals",             "city": "Delhi",     "titles": 0},
    {"team_id": 6, "name": "Rajasthan Royals",           "city": "Jaipur",    "titles": 2},
    {"team_id": 7, "name": "Sunrisers Hyderabad",        "city": "Hyderabad", "titles": 1},
    {"team_id": 8, "name": "Punjab Kings",               "city": "Mohali",    "titles": 0},
])

# --- Players ---
players = pd.DataFrame([
    {"player_id":  1, "name": "Rohit Sharma",      "role": "Batsman",        "nationality": "Indian",       "batting_style": "Right-hand"},
    {"player_id":  2, "name": "Virat Kohli",        "role": "Batsman",        "nationality": "Indian",       "batting_style": "Right-hand"},
    {"player_id":  3, "name": "MS Dhoni",           "role": "Wicket-keeper",  "nationality": "Indian",       "batting_style": "Right-hand"},
    {"player_id":  4, "name": "Hardik Pandya",      "role": "All-rounder",    "nationality": "Indian",       "batting_style": "Right-hand"},
    {"player_id":  5, "name": "Jasprit Bumrah",     "role": "Bowler",         "nationality": "Indian",       "batting_style": "Right-hand"},
    {"player_id":  6, "name": "Ravindra Jadeja",    "role": "All-rounder",    "nationality": "Indian",       "batting_style": "Left-hand"},
    {"player_id":  7, "name": "KL Rahul",           "role": "Batsman",        "nationality": "Indian",       "batting_style": "Right-hand"},
    {"player_id":  8, "name": "Suryakumar Yadav",   "role": "Batsman",        "nationality": "Indian",       "batting_style": "Right-hand"},
    {"player_id":  9, "name": "Pat Cummins",        "role": "Bowler",         "nationality": "Australian",   "batting_style": "Right-hand"},
    {"player_id": 10, "name": "Andre Russell",      "role": "All-rounder",    "nationality": "West Indian",  "batting_style": "Right-hand"},
    {"player_id": 11, "name": "AB de Villiers",     "role": "Batsman",        "nationality": "South African","batting_style": "Right-hand"},
    {"player_id": 12, "name": "Faf du Plessis",     "role": "Batsman",        "nationality": "South African","batting_style": "Right-hand"},
    {"player_id": 13, "name": "Jos Buttler",        "role": "Batsman",        "nationality": "English",      "batting_style": "Right-hand"},
    {"player_id": 14, "name": "David Warner",       "role": "Batsman",        "nationality": "Australian",   "batting_style": "Left-hand"},
    {"player_id": 15, "name": "Rashid Khan",        "role": "Bowler",         "nationality": "Afghan",       "batting_style": "Right-hand"},
])

# --- Games ---
games = pd.DataFrame([
    {"match_id":  1, "season": 2023, "venue": "Wankhede Stadium",     "winner_team_id": 1, "mom_player_id":  1},
    {"match_id":  2, "season": 2023, "venue": "Chepauk",              "winner_team_id": 2, "mom_player_id":  3},
    {"match_id":  3, "season": 2023, "venue": "Chinnaswamy",          "winner_team_id": 3, "mom_player_id":  2},
    {"match_id":  4, "season": 2023, "venue": "Eden Gardens",         "winner_team_id": 4, "mom_player_id": 10},
    {"match_id":  5, "season": 2022, "venue": "Wankhede Stadium",     "winner_team_id": 1, "mom_player_id":  5},
    {"match_id":  6, "season": 2022, "venue": "Chepauk",              "winner_team_id": 2, "mom_player_id":  6},
    {"match_id":  7, "season": 2022, "venue": "Chinnaswamy",          "winner_team_id": 3, "mom_player_id": 11},
    {"match_id":  8, "season": 2024, "venue": "Eden Gardens",         "winner_team_id": 4, "mom_player_id":  9},
    {"match_id":  9, "season": 2024, "venue": "Sawai Mansingh",       "winner_team_id": 6, "mom_player_id": 13},
    {"match_id": 10, "season": 2024, "venue": "Rajiv Gandhi Intl",    "winner_team_id": 7, "mom_player_id": 14},
])

# --- PLAYS_FOR edges ---
plays_for = pd.DataFrame([
    {"player_id":  1, "team_id": 1},  # Rohit Sharma → Mumbai Indians
    {"player_id":  4, "team_id": 1},  # Hardik Pandya → Mumbai Indians
    {"player_id":  5, "team_id": 1},  # Jasprit Bumrah → Mumbai Indians
    {"player_id":  8, "team_id": 1},  # Suryakumar Yadav → Mumbai Indians
    {"player_id":  3, "team_id": 2},  # MS Dhoni → Chennai Super Kings
    {"player_id":  6, "team_id": 2},  # Ravindra Jadeja → Chennai Super Kings
    {"player_id":  2, "team_id": 3},  # Virat Kohli → Royal Challengers Bengaluru
    {"player_id": 11, "team_id": 3},  # AB de Villiers → Royal Challengers Bengaluru
    {"player_id": 12, "team_id": 3},  # Faf du Plessis → Royal Challengers Bengaluru
    {"player_id": 10, "team_id": 4},  # Andre Russell → Kolkata Knight Riders
    {"player_id":  9, "team_id": 4},  # Pat Cummins → Kolkata Knight Riders
    {"player_id":  7, "team_id": 5},  # KL Rahul → Delhi Capitals
    {"player_id": 13, "team_id": 6},  # Jos Buttler → Rajasthan Royals
    {"player_id": 15, "team_id": 6},  # Rashid Khan → Rajasthan Royals
    {"player_id": 14, "team_id": 7},  # David Warner → Sunrisers Hyderabad
])

# --- PLAYED_IN edges (with runs and wickets as properties) ---
played_in = pd.DataFrame([
    {"player_id":  1, "match_id":  1, "runs": 72, "wickets": 0},  # Rohit, match 1
    {"player_id":  1, "match_id":  5, "runs": 45, "wickets": 0},  # Rohit, match 5
    {"player_id":  5, "match_id":  1, "runs":  4, "wickets": 2},  # Bumrah, match 1
    {"player_id":  5, "match_id":  5, "runs":  0, "wickets": 3},  # Bumrah, match 5
    {"player_id":  4, "match_id":  1, "runs": 38, "wickets": 1},  # Hardik, match 1
    {"player_id":  8, "match_id":  1, "runs": 55, "wickets": 0},  # SKY, match 1
    {"player_id":  3, "match_id":  2, "runs": 28, "wickets": 0},  # Dhoni, match 2
    {"player_id":  6, "match_id":  2, "runs": 22, "wickets": 2},  # Jadeja, match 2
    {"player_id":  6, "match_id":  6, "runs": 35, "wickets": 3},  # Jadeja, match 6
    {"player_id":  3, "match_id":  6, "runs": 19, "wickets": 0},  # Dhoni, match 6
    {"player_id":  2, "match_id":  3, "runs": 82, "wickets": 0},  # Kohli, match 3
    {"player_id":  2, "match_id":  7, "runs": 91, "wickets": 0},  # Kohli, match 7
    {"player_id": 11, "match_id":  3, "runs": 65, "wickets": 0},  # AB, match 3
    {"player_id": 11, "match_id":  7, "runs": 48, "wickets": 0},  # AB, match 7
    {"player_id": 10, "match_id":  4, "runs": 67, "wickets": 2},  # Russell, match 4
    {"player_id":  9, "match_id":  4, "runs":  5, "wickets": 3},  # Cummins, match 4
    {"player_id":  9, "match_id":  8, "runs":  8, "wickets": 4},  # Cummins, match 8
    {"player_id": 10, "match_id":  8, "runs": 54, "wickets": 1},  # Russell, match 8
    {"player_id": 13, "match_id":  9, "runs": 95, "wickets": 0},  # Buttler, match 9
    {"player_id": 15, "match_id":  9, "runs":  2, "wickets": 3},  # Rashid, match 9
    {"player_id": 14, "match_id": 10, "runs": 88, "wickets": 0},  # Warner, match 10
])

# --- WON edges ---
won = pd.DataFrame([
    {"team_id": 1, "match_id":  1},  # MI won match 1
    {"team_id": 2, "match_id":  2},  # CSK won match 2
    {"team_id": 3, "match_id":  3},  # RCB won match 3
    {"team_id": 4, "match_id":  4},  # KKR won match 4
    {"team_id": 1, "match_id":  5},  # MI won match 5
    {"team_id": 2, "match_id":  6},  # CSK won match 6
    {"team_id": 3, "match_id":  7},  # RCB won match 7
    {"team_id": 4, "match_id":  8},  # KKR won match 8
    {"team_id": 6, "match_id":  9},  # RR won match 9
    {"team_id": 7, "match_id": 10},  # SRH won match 10
])

print(f"Teams: {len(teams)}, Players: {len(players)}, Games: {len(games)}")
print(f"PLAYS_FOR edges: {len(plays_for)}, PLAYED_IN edges: {len(played_in)}, WON edges: {len(won)}")

Teams: 8, Players: 15, Games: 10
PLAYS_FOR edges: 15, PLAYED_IN edges: 21, WON edges: 10


## Step 2 — Save to Parquet

In [3]:
# ── Step 3 │ Save to Parquet ───────────────────────────────────────────────
# Services : (local filesystem — no network)
# ──────────────────────────────────────────────────────────────────────────
base = Path("/tmp/ipl-graph")
for p in ["nodes/Team", "nodes/Player", "nodes/Game",
          "edges/PLAYS_FOR", "edges/PLAYED_IN", "edges/WON"]:
    (base / p).mkdir(parents=True, exist_ok=True)

teams.to_parquet(base / "nodes/Team/data.parquet",          index=False)
players.to_parquet(base / "nodes/Player/data.parquet",      index=False)
games.to_parquet(base / "nodes/Game/data.parquet",       index=False)
plays_for.to_parquet(base / "edges/PLAYS_FOR/data.parquet", index=False)
played_in.to_parquet(base / "edges/PLAYED_IN/data.parquet", index=False)
won.to_parquet(base / "edges/WON/data.parquet",             index=False)

print("✅ Parquet files written to /tmp/ipl-graph/")

✅ Parquet files written to /tmp/ipl-graph/


## Step 3 — Create Mapping

In [4]:
# ── Step 4 │ Create Mapping ────────────────────────────────────────────────
# Services : Control Plane (localhost:8081)
# ──────────────────────────────────────────────────────────────────────────
mapping = post("/api/mappings", {
    "name": "IPL T20 Graph",
    "description": "IPL T20 cricket teams, players, and matches — synthetic dataset",
    "node_definitions": [
        {
            "label": "Team",
            "sql": "SELECT team_id, name, city, titles FROM placeholder",
            "primary_key": {"name": "team_id", "type": "INT64"},
            "properties": [
                {"name": "name",   "type": "STRING"},
                {"name": "city",   "type": "STRING"},
                {"name": "titles", "type": "INT64"},
            ]
        },
        {
            "label": "Player",
            "sql": "SELECT player_id, name, role, nationality, batting_style FROM placeholder",
            "primary_key": {"name": "player_id", "type": "INT64"},
            "properties": [
                {"name": "name",          "type": "STRING"},
                {"name": "role",          "type": "STRING"},
                {"name": "nationality",   "type": "STRING"},
                {"name": "batting_style", "type": "STRING"},
            ]
        },
        {
            "label": "Game",
            "sql": "SELECT match_id, season, venue, winner_team_id, mom_player_id FROM placeholder",
            "primary_key": {"name": "match_id", "type": "INT64"},
            "properties": [
                {"name": "season",         "type": "INT64"},
                {"name": "venue",          "type": "STRING"},
                {"name": "winner_team_id", "type": "INT64"},
                {"name": "mom_player_id",  "type": "INT64"},
            ]
        },
    ],
    "edge_definitions": [
        {
            "type": "PLAYS_FOR",
            "from_node": "Player",
            "to_node": "Team",
            "sql": "SELECT player_id, team_id FROM placeholder",
            "from_key": "player_id",
            "to_key": "team_id",
            "properties": []
        },
        {
            "type": "PLAYED_IN",
            "from_node": "Player",
            "to_node": "Game",
            "sql": "SELECT player_id, match_id, runs, wickets FROM placeholder",
            "from_key": "player_id",
            "to_key": "match_id",
            "properties": [
                {"name": "runs",    "type": "INT64"},
                {"name": "wickets", "type": "INT64"},
            ]
        },
        {
            "type": "WON",
            "from_node": "Team",
            "to_node": "Game",
            "sql": "SELECT team_id, match_id FROM placeholder",
            "from_key": "team_id",
            "to_key": "match_id",
            "properties": []
        },
    ]
})

mapping_id = mapping["data"]["id"]
print(f"✅ Mapping created: id={mapping_id}")

✅ Mapping created: id=5


## Step 4 — Create Instance

In [5]:
# ── Step 5 │ Create Instance + Bypass Export Worker ───────────────────────
# Services : Control Plane (localhost:8081)  ·  PostgreSQL (postgres:5432)
# ──────────────────────────────────────────────────────────────────────────
import psycopg2

instance = post("/api/instances", {
    "mapping_id": mapping_id,
    "wrapper_type": "falkordb",
    "name": "IPL T20 Instance",
    "ttl": "PT4H",
})
instance_id = instance["data"]["id"]
snapshot_id = instance["data"]["snapshot_id"]
print(f"✅ Instance created: id={instance_id}, snapshot_id={snapshot_id}")

# Immediately delete any export_jobs and mark the snapshot as ready via psycopg2.
# This prevents the export worker from failing on a snapshot it should not process.
conn = psycopg2.connect(
    host="localhost", port=5432,
    dbname="grapholap", user="grapholap", password="grapholap"
)
conn.autocommit = True
cur = conn.cursor()

cur.execute(
    "DELETE FROM export_jobs WHERE snapshot_id = %s",
    (snapshot_id,)
)
print(f"  export_jobs deleted: {cur.rowcount} row(s)")

cur.execute(
    "UPDATE snapshots SET status = 'ready' WHERE id = %s",
    (snapshot_id,)
)
print(f"  snapshot marked ready: {cur.rowcount} row(s)")

cur.close()
conn.close()
print("✅ DB patched — export worker will not interfere")

✅ Instance created: id=4, snapshot_id=4
  export_jobs deleted: 6 row(s)
  snapshot marked ready: 1 row(s)
✅ DB patched — export worker will not interfere


## Step 5 — Upload Parquet files to GCS

In [6]:
# ── Step 6 │ Upload Parquet to GCS ────────────────────────────────────────
# Services : localhost (localhost:4443)
# ──────────────────────────────────────────────────────────────────────────
# Upload parquet files directly to fake-gcs via HTTP API
import requests as _req

BUCKET = "graph-olap-local-dev"
GCS    = "http://localhost:4443"
OWNER  = "demo@example.com"
PREFIX = f"{OWNER}/{mapping_id}/v1/{snapshot_id}"
print(f"Uploading to gs://{BUCKET}/{PREFIX}/")

# Ensure bucket exists in fake-gcs (in-memory — recreated after each pod restart)
_br = _req.post(f"{GCS}/storage/v1/b", json={"name": BUCKET},
               headers={"Content-Type": "application/json"})
if _br.status_code not in (200, 201, 409):
    _br.raise_for_status()

files = [
    ("nodes/Team/data.parquet",     f"{PREFIX}/nodes/Team/data.parquet"),
    ("nodes/Player/data.parquet",   f"{PREFIX}/nodes/Player/data.parquet"),
    ("nodes/Game/data.parquet",    f"{PREFIX}/nodes/Game/data.parquet"),
    ("edges/PLAYS_FOR/data.parquet",f"{PREFIX}/edges/PLAYS_FOR/data.parquet"),
    ("edges/PLAYED_IN/data.parquet",f"{PREFIX}/edges/PLAYED_IN/data.parquet"),
    ("edges/WON/data.parquet",      f"{PREFIX}/edges/WON/data.parquet"),
]

for local, remote in files:
    file_path = base / local
    with open(file_path, "rb") as f:
        data = f.read()
    url  = f"{GCS}/upload/storage/v1/b/{BUCKET}/o?uploadType=media&name={remote}"
    resp = _req.post(url, data=data, headers={"Content-Type": "application/octet-stream"})
    if resp.status_code in (200, 201):
        print(f"  ✅ {remote}")
    else:
        print(f"  ❌ {remote}: {resp.status_code} {resp.text[:200]}")

Uploading to gs://graph-olap-local-dev/demo@example.com/5/v1/4/
  ✅ demo@example.com/5/v1/4/nodes/Team/data.parquet
  ✅ demo@example.com/5/v1/4/nodes/Player/data.parquet
  ✅ demo@example.com/5/v1/4/nodes/Game/data.parquet
  ✅ demo@example.com/5/v1/4/edges/PLAYS_FOR/data.parquet
  ✅ demo@example.com/5/v1/4/edges/PLAYED_IN/data.parquet
  ✅ demo@example.com/5/v1/4/edges/WON/data.parquet


## Step 6 — Mark snapshot as ready

In [7]:
# ── Step 6b │ Verify Snapshot Status ───────────────────────────────
# Services : PostgreSQL (postgres:5432)
# ──────────────────────────────────────────────────────────────────────
# Snapshot was already marked ready in Step 4 via psycopg2.
print(f"Snapshot {snapshot_id} status: ready (patched via psycopg2)")

Snapshot 4 status: ready (patched via psycopg2)


## Step 7 — Wait for instance to be running

In [ ]:
# ── Step 7 │ Wait for Instance ─────────────────────────────────────
# Services : Control Plane (localhost:8081)
# ──────────────────────────────────────────────────────────────────────
print("Polling... (reconciliation loop runs every ~30s)")
start = time.time()
while True:
    elapsed = int(time.time() - start)
    inst    = get(f"/api/instances/{instance_id}")
    status  = inst["data"]["status"]
    print(f"  [{elapsed:3d}s] instance={status}")
    if status == "running":
        break
    if status in ("stopped", "failed"):
        raise RuntimeError(f"Instance ended with status: {status}")
    time.sleep(10)

print(f"\n🎉 IPL T20 graph is running! FalkorDB pod ready.")

Polling... (reconciliation loop runs every ~30s)
  [  0s] instance=waiting_for_snapshot
  [ 10s] instance=starting
  [ 20s] instance=starting
  [ 30s] instance=starting
  [ 40s] instance=starting
  [ 50s] instance=starting
  [ 60s] instance=starting
  [ 70s] instance=starting
  [ 80s] instance=starting
  [ 90s] instance=starting
  [100s] instance=starting


## Step 8 — Query the IPL Graph

In [ ]:
# ── Step 8 │ Connect & Query Graph ─────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
inst_data = get(f"/api/instances/{instance_id}")["data"]
pod_name = inst_data["pod_name"]
print(f"Wrapper: http://{pod_name}:8000")

def cypher(q, params=None):
    body = {"query": q}
    if params:
        body["parameters"] = params
    for attempt in range(10):
        try:
            r = requests.post(f"http://{pod_name}:8000/query",
                              headers={"Content-Type": "application/json"}, json=body)
            r.raise_for_status()
            return r.json()["rows"]
        except Exception as e:
            if attempt < 9:
                print(f"  Wrapper not ready yet, retrying in 3s... ({attempt+1}/10)")
                time.sleep(3)
            else:
                raise

# --- Node counts ---
print("=== Graph loaded — Node counts ===")
for row in cypher("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt ORDER BY cnt DESC"):
    print(f"  {str(row[0]):20s}: {row[1]:,}")

In [ ]:
# ── Step 8a │ Cypher — Top Run Scorers ─────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# --- All-time top run scorers ---
print("=== All-time top run scorers ===")
results = cypher("""
    MATCH (p:Player)-[r:PLAYED_IN]->(:Game)
    RETURN p.name AS player, sum(r.runs) AS total_runs
    ORDER BY total_runs DESC
""")
for i, row in enumerate(results, 1):
    print(f"  {i:2d}. {row[0]:25s}  {int(row[1]):4d} runs")

In [ ]:
# ── Step 8b │ Cypher — Best Bowlers ────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# --- Best bowlers by total wickets ---
print("=== Best bowlers (total wickets) ===")
results = cypher("""
    MATCH (p:Player)-[r:PLAYED_IN]->(:Game)
    RETURN p.name AS player, sum(r.wickets) AS total_wickets
    ORDER BY total_wickets DESC
""")
for i, row in enumerate(results, 1):
    if row[1] > 0:
        print(f"  {i:2d}. {row[0]:25s}  {int(row[1]):2d} wickets")

In [ ]:
# ── Step 8c │ Cypher — Team Wins ───────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# --- Teams with most wins ---
print("=== Teams with most wins ===")
results = cypher("""
    MATCH (t:Team)-[:WON]->(m:Game)
    RETURN t.name AS team, count(m) AS wins
    ORDER BY wins DESC
""")
for row in results:
    print(f"  {row[0]:35s}  {row[1]} win(s)")

In [ ]:
# ── Step 8d │ Cypher — Man of the Match ────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# --- Man of the Match awards ---
# MOM is stored as a property on Match (mom_player_id), not as a relationship.
# Join via PLAYED_IN to identify the award winner per match.
print("=== Man of the Match awards ===")
results = cypher("""
    MATCH (p:Player)-[:PLAYED_IN]->(m:Game)
    WHERE m.mom_player_id = p.player_id
    RETURN p.name AS player, count(m) AS mom_awards
    ORDER BY mom_awards DESC
""")
for row in results:
    print(f"  {row[0]:25s}  {row[1]} award(s)")

In [ ]:
# ── Step 8e │ Cypher — Players by Nationality ──────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# --- Players by nationality ---
print("=== Players by nationality ===")
results = cypher("""
    MATCH (p:Player)
    RETURN p.nationality AS nationality, count(p) AS players
    ORDER BY players DESC
""")
for row in results:
    print(f"  {row[0]:20s}  {row[1]} player(s)")

In [ ]:
# ── Step 8f │ Cypher — Players in Team Wins ────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# --- Players who played in matches won by their own team ---
# FalkorDB does not support undirected shortestPath.
# Use explicit directional traversal instead.
print("=== Players who featured in their team's wins ===")
results = cypher("""
    MATCH (p:Player)-[:PLAYS_FOR]->(t:Team)-[:WON]->(m:Game)
    MATCH (p)-[:PLAYED_IN]->(m)
    RETURN p.name AS player, t.name AS team, count(m) AS wins_played_in
    ORDER BY wins_played_in DESC
""")
for row in results:
    print(f"  {row[0]:25s}  ({row[1]:35s})  {row[2]} win(s)")

## Step 9 — Visualise (PyVis)

In [ ]:
# ── Step 9 │ Visualise Graph (PyVis) ───────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)  ·  PyVis (local)
# ──────────────────────────────────────────────────────────────────────
from pyvis.network import Network
from IPython.display import IFrame

net = Network(height="650px", width="100%", bgcolor="#0d1117", font_color="white", notebook=True)
net.toggle_physics(True)

# Color scheme
colors = {
    "Team":   "#f0a500",  # gold
    "Player": "#0f7ddb",  # blue
    "Game":  "#3cb371",  # green
}

# Fetch Players + their teams
player_rows = cypher("""
    MATCH (p:Player)-[:PLAYS_FOR]->(t:Team)
    RETURN p.player_id AS p_id, p.name AS p_name, p.role AS p_role,
           t.team_id AS t_id, t.name AS t_name
""")

for row in player_rows:
    p_id, p_name, p_role, t_id, t_name = row
    net.add_node(f"p_{p_id}", label=p_name,  color=colors["Player"], size=18,
                 title=f"Player: {p_name}\nRole: {p_role}")
    net.add_node(f"t_{t_id}", label=t_name,  color=colors["Team"],   size=30,
                 title=f"Team: {t_name}", shape="star")
    net.add_edge(f"p_{p_id}", f"t_{t_id}", title="PLAYS_FOR", color="#aaa")

# Fetch Matches won by teams
match_rows = cypher("""
    MATCH (t:Team)-[:WON]->(m:Game)
    RETURN t.team_id AS t_id, m.match_id AS m_id,
           m.season AS season, m.venue AS venue
""")

for row in match_rows:
    t_id, m_id, season, venue = row
    net.add_node(f"m_{m_id}", label=f"{season}\n{venue[:12]}", color=colors["Game"],
                 size=20, title=f"Match {m_id}: {venue} ({season})")
    net.add_edge(f"t_{t_id}", f"m_{m_id}", title="WON", color="#f0a500", width=2)

net.show("ipl_graph.html")
IFrame("ipl_graph.html", width="100%", height=670)

## Cleanup (optional)

In [ ]:
# ── Step 10 │ Cleanup ──────────────────────────────────────────────
# Services : Control Plane (localhost:8081)
# ──────────────────────────────────────────────────────────────────────
# Clean up — delete the instance to free the wrapper pod memory
try:
    delete(f"/api/instances/{instance_id}")
    print(f"Instance {instance_id} deleted — wrapper pod will be removed")
except Exception as e:
    print(f"Instance {instance_id} already gone or not found: {e}")